In [0]:
-- ============================================
-- 🔍 VISTA BASE: MARKET OVERVIEW
-- ============================================
-- Esta vista consolida todas las dimensiones y hechos en un solo lugar
-- Sirve como base para las demás vistas analíticas

CREATE OR REPLACE VIEW prueba_api.semantic.vw_market_overview
COMMENT 'Vista consolidada del mercado laboral con todas las dimensiones y métricas'
AS
SELECT 
    -- Hechos
    f.job_id,
    f.job_title,
    f.job_description,
    f.job_is_remote,
    f.job_posted_at_timestamp,
    f.job_posted_at_datetime_utc,
    f.job_apply_link,
    
    -- Dimensión: Empleador
    e.employer_sk,
    e.employer_name,
    e.employer_logo,
    e.employer_website,
    
    -- Dimensión: Ubicación
    l.location_sk,
    l.job_city,
    l.job_state,
    l.job_country,
    
    -- Dimensión: Tipo de Empleo
    et.employment_type_sk,
    et.job_employment_type,
    et.employment_type_es,
    
    -- Campos derivados útiles
    CASE 
        WHEN f.job_is_remote = true THEN 'Remoto'
        ELSE 'Presencial'
    END AS modalidad_trabajo,
    
    -- Metadata
    f.processed_at AS fecha_carga
    
FROM prueba_api.gold.fact_jobs f
INNER JOIN prueba_api.gold.dim_employer e 
    ON f.employer_sk = e.employer_sk
INNER JOIN prueba_api.gold.dim_location l 
    ON f.location_sk = l.location_sk
INNER JOIN prueba_api.gold.dim_employment_type et 
    ON f.employment_type_sk = et.employment_type_sk;

In [0]:
-- ============================================
-- EJEMPLO: ANÁLISIS AD-HOC CON VISTA BASE
-- ============================================
-- La vista vw_market_overview permite crear análisis personalizados
-- sin necesidad de escribir los JOINs manualmente

-- Ejemplo: Top 10 ciudades con más ofertas remotas
SELECT 
    job_city AS ciudad,
    job_state AS estado,
    job_country AS pais,
    COUNT(*) AS ofertas_remotas,
    COUNT(DISTINCT employer_name) AS empleadores,
    COUNT(DISTINCT job_employment_type) AS tipos_empleo
FROM prueba_api.semantic.vw_market_overview
WHERE 
    job_is_remote = true 
    AND job_city IS NOT NULL
GROUP BY job_city, job_state, job_country
ORDER BY ofertas_remotas DESC
LIMIT 10;